# CMSC 173 &middot; Module 8 &middot; Example 1
## Nine days of tennis, by hand

The lecture's running example: nine days, three weather features, and whether we played. Naive Bayes and the decision tree are both computed here with nothing but counting.

**How to use this notebook.** Runtime &rarr; Run all. Every number printed below is on the
slides; change something and see what moves.

In [ ]:
import numpy as np
import pandas as pd

days = pd.DataFrame({
    "Outlook":  ["Sunny", "Sunny", "Overcast", "Rainy", "Rainy", "Rainy", "Overcast", "Sunny", "Sunny"],
    "Temp":     ["Hot", "Hot", "Hot", "Mild", "Cool", "Cool", "Cool", "Mild", "Cool"],
    "Humidity": ["High", "High", "High", "High", "Normal", "Normal", "Normal", "High", "Normal"],
    "Play":     ["No", "No", "Yes", "Yes", "Yes", "No", "Yes", "No", "Yes"]})
pd.crosstab(days["Outlook"], days["Play"])     # one block of the count table

**Naive Bayes for a Sunny, Cool, High day.** Prior &times; one likelihood per feature, each read off the counts.

In [ ]:
new_day = {"Outlook": "Sunny", "Temp": "Cool", "Humidity": "High"}
score = {}
for c in ("Yes", "No"):
    rows = days[days["Play"] == c]
    s = len(rows) / len(days)                       # prior: 5/9 or 4/9
    for feature, value in new_day.items():
        s *= (rows[feature] == value).mean()        # likelihood, e.g. P(Sunny | Yes) = 1/5
    score[c] = s
total = sum(score.values())
print({c: round(s, 4) for c, s in score.items()})   # Yes 0.0267, No 0.0625
print({c: round(s / total, 2) for c, s in score.items()})   # Yes 0.3, No 0.7

**Laplace smoothing.** An Overcast day was never a No day, so without smoothing No scores exactly 0. Add 1 to every count:

In [ ]:
def likelihood(rows, feature, value, alpha=1):
    V = days[feature].nunique()                     # 3 outlooks, 3 temperatures, 2 humidities
    return ((rows[feature] == value).sum() + alpha) / (len(rows) + alpha * V)

overcast_day = {"Outlook": "Overcast", "Temp": "Hot", "Humidity": "High"}
for c in ("Yes", "No"):
    rows = days[days["Play"] == c]
    s = len(rows) / len(days)
    for feature, value in overcast_day.items():
        s *= likelihood(rows, feature, value)
    print(c, round(s, 4))                           # Yes 0.0223, No 0.0181

**A feature that is a number: Gaussian naive Bayes.** Constructed temperatures: Yes days 22, 24, 26 &deg;C; No days 30, 32, 34 &deg;C. Today is 27 &deg;C.

In [ ]:
def bell(x, values):
    mu, var = np.mean(values), np.var(values)       # np.var divides by n, as on the slide
    return np.exp(-(x - mu) ** 2 / (2 * var)) / np.sqrt(2 * np.pi * var)

yes, no = bell(27, [22, 24, 26]), bell(27, [30, 32, 34])
print(round(yes, 4), round(no, 5), "P(Yes | 27) =", round(yes / (yes + no), 3))   # 0.0452 0.00225 0.953

**The same in scikit-learn.** `CategoricalNB` smooths with `alpha=1` by default, so it prints 0.619, not 0.70.

In [ ]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder

enc = OrdinalEncoder()
X = enc.fit_transform(days[["Outlook", "Temp", "Humidity"]])
nb = CategoricalNB(alpha=1).fit(X, days["Play"])
query = pd.DataFrame([["Sunny", "Cool", "High"]], columns=["Outlook", "Temp", "Humidity"])
print(nb.classes_, nb.predict_proba(enc.transform(query)).round(3))   # ['No' 'Yes'] [[0.619 0.381]]

**The tree's first question, by hand.** Entropy of the nine days, then the information gain of each feature (Part 6 of the lecture).

In [ ]:
def entropy(labels):
    p = labels.value_counts(normalize=True)
    return -(p * np.log2(p)).sum()

root = entropy(days["Play"])
print("root entropy:", round(root, 3))              # 0.991
for f in ("Outlook", "Temp", "Humidity"):
    after = sum(len(g) / len(days) * entropy(g["Play"]) for _, g in days.groupby(f))
    print(f, "gain:", round(root - after, 3))       # Outlook 0.324, Temp 0.102, Humidity 0.091

**Try it.** Change day 6 from No to Yes. Which feature has the highest gain now, and what does naive Bayes say about the Sunny, Cool, High day?